In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("processed_dataset.csv",low_memory=False) #license column contains different types of values,so we have to use low_mwemory=false.

In [3]:
print("Invalid minimum nights:")
print((df["minimum_nights"] < 1).sum())

print("\nInvalid availability:")
print(
    (~df["availability_365"].between(0, 365)).sum()
)

print("\nPrice equal to 524:")
print((df["price"] == 524).sum())

print("\nPrice and service fee correlation:")
print(df[["price", "service_fee"]].corr())

Invalid minimum nights:


KeyError: 'minimum_nights'

In [4]:
print(df.columns.tolist())#check the column names

['host_identity_verified', 'neighbourhood group', 'lat', 'long', 'instant_bookable', 'cancellation_policy', 'room type', 'price', 'service fee', 'minimum nights', 'number of reviews', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules', 'Property Age']


In [5]:
df.columns=df.columns.str.strip() #remove the extra spaces
for col in df.columns:
    print(repr(col))

'host_identity_verified'
'neighbourhood group'
'lat'
'long'
'instant_bookable'
'cancellation_policy'
'room type'
'price'
'service fee'
'minimum nights'
'number of reviews'
'reviews per month'
'review rate number'
'calculated host listings count'
'availability 365'
'house_rules'
'Property Age'


In [6]:
# Check invalid values using your actual column names

print("Invalid minimum nights:")
print((df["minimum nights"] < 1).sum())

print("\nInvalid availability:")
print(
    (~df["availability 365"].between(0, 365)).sum()
)

print("\nPrice equal to 524:")
print((df["price"] == 524).sum())

print("\nPrice and service fee correlation:")
print(df[["price", "service fee"]].corr())

Invalid minimum nights:
9

Invalid availability:
2706

Price equal to 524:
11547

Price and service fee correlation:
                price  service fee
price        1.000000     0.748473
service fee  0.748473     1.000000


In [7]:
import numpy as np #fix the price issues

# Work on a copy
df = df.copy()

# Find prices equal to 524
mask = df["price"].eq(524)

print("Price = 524 before:", int(mask.sum()))

# Create donor pool
donor_prices = df.loc[
    df["price"].notna() & ~mask,
    "price"
].to_numpy()

if mask.sum() > 0 and len(donor_prices) == 0:
    raise ValueError("No donor prices available.")

# Reproducible replacement
rng = np.random.default_rng(42)

if mask.sum() > 0:
    df.loc[mask, "price"] = rng.choice(
        donor_prices,
        size=int(mask.sum()),
        replace=True
    )

print("Price = 524 after:", int(df["price"].eq(524).sum()))

Price = 524 before: 11547
Price = 524 after: 0


In [8]:
before = len(df) #clean the invalid rows

df = df[
    (df["minimum nights"] >= 1) &
    (df["availability 365"].between(0, 365))
].copy()

print("Rows before cleaning:", before)
print("Rows after cleaning:", len(df))
print("Rows removed:", before - len(df))

Rows before cleaning: 65319
Rows after cleaning: 62605
Rows removed: 2714


In [9]:
# Check the cleaned dataset

print("Dataset shape:", df.shape)

print("\nInvalid minimum nights:",
      (df["minimum nights"] < 1).sum())

print("\nInvalid availability:",
      (~df["availability 365"].between(0, 365)).sum())

print("\nRemaining price = 524:",
      df["price"].eq(524).sum())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nData types:")
print(df.dtypes)

Dataset shape: (62605, 17)

Invalid minimum nights: 0

Invalid availability: 0

Remaining price = 524: 0

Missing values:
host_identity_verified            0
minimum nights                    0
house_rules                       0
availability 365                  0
calculated host listings count    0
review rate number                0
reviews per month                 0
number of reviews                 0
service fee                       0
neighbourhood group               0
dtype: int64

Data types:
host_identity_verified             object
neighbourhood group                object
lat                               float64
long                              float64
instant_bookable                     bool
cancellation_policy                object
room type                          object
price                               int64
service fee                         int64
minimum nights                      int64
number of reviews                 float64
reviews per month             

In [12]:
df.to_csv("final_processed_dataset1.csv",index=False)